# Hebrew Contextualized Embeddings

Generates XLM-RoBERTa contextual embeddings for the Hebrew column of the podcast transcript.

**Input:**
- `translated_podcast_transcript_filtered.csv` — timestamped content words (he column)
- `podcast_sentences_he.csv` — full Hebrew sentences for context

**Output:**
- `he_contextual_aligned_embeddings.csv` — (1735, 768) embedding matrix
- `he_contextual_quality_flags.csv` — which words got full context vs fallback

**Key differences from English pipeline:**
- Target words come from `he` column, space-separated (no underscores)
- Hebrew-specific normalization strips niqqud (vowel marks U+05B0–U+05C7)
- Sentences file has 3 columns: sentence_id, sentence_en, sentence_he

## 1. Load Data

In [1]:
import pandas as pd
import torch
import numpy as np
import unicodedata
import csv
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

# --- Load word-level data (same file as English) ---
print("Loading datasets...")
word_level_df = pd.read_csv('../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv')

# --- Load Hebrew sentences ---
# File has columns: sentence_id, sentence_en, sentence_he
# Use csv.reader to handle commas inside quoted fields correctly
he_sentences = []
with open('../data/sentences/podcast_sentences_he.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        if len(row) >= 3:
            he_sentences.append(row[2])  # sentence_he is column index 2

print(f"Target words     : {len(word_level_df)}")
print(f"Hebrew sentences : {len(he_sentences)}")
print(f"\nFirst 5 Hebrew sentences:")
for i, s in enumerate(he_sentences[:5]):
    print(f"  [{i}] '{s}'")

print(f"\nFirst 5 Hebrew target words:")
print(word_level_df[['start', 'en', 'he']].head())

/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading datasets...
Target words     : 1735
Hebrew sentences : 402

First 5 Hebrew sentences:
  [0] 'מערכה ראשונה, קוף באמצע.'
  [1] 'אז יש כמה מקומות שבעלי חיים כמעט ולא הולכים אליהם, מקומות שעוצבו על ידי בני אדם לבני אדם.'
  [2] 'המעשה הזה מסתיים במקום כזה, אבל הוא מתחיל בערך הכי רחוק משם שאפשר להגיע.'
  [3] 'דנה צ'יביס מסבירה.'
  [4] 'הסיפור שלנו מתחיל עמוק ביערות הגשם של אינדונזיה על אי בשם Sulawesi.'

First 5 Hebrew target words:
   start       en         he
0  3.710      act      מערכה
1  4.651   monkey        קוף
2  5.151   middle       אמצע
3  7.072   places     מקומות
4  7.542  animals  בעלי חיים


## 2. Helpers

In [2]:
# Common Hebrew prefix combinations (longest first for correct stripping order)
HEBREW_PREFIXES = sorted([
    'וה', 'ובה', 'ומה', 'בה', 'לה', 'כה', 'מה', 'שה',
    'כש', 'מש', 'לכ', 'בכ', 'שב', 'שכ', 'שמ', 'של',
    'ו', 'ב', 'ל', 'ה', 'כ', 'מ', 'ש', 'כ'
], key=len, reverse=True)

HEBREW_SUFFIXES = sorted([
    'ים', 'ות', 'יים',  # plural
    'תי', 'תם', 'תן',   # past tense conjugations
    'נו', 'כם', 'כן', 'הם', 'הן',  # possessive/pronoun suffixes
    'ני', 'ך', 'ו', 'ה', 'י',      # shorter possessives + feminine
    'ת',                             # feminine/construct
], key=len, reverse=True)


def get_hebrew_stem(word):
    """
    Returns candidate stems by stripping prefixes and/or suffixes.
    Requires stem to be at least 2 characters.
    """
    candidates = set()
    candidates.add(word)
    
    # Strip prefixes only
    prefix_stripped = [word]
    for prefix in HEBREW_PREFIXES:
        if word.startswith(prefix) and len(word) > len(prefix) + 1:
            prefix_stripped.append(word[len(prefix):])
            candidates.add(word[len(prefix):])
    
    # Strip suffixes only
    for suffix in HEBREW_SUFFIXES:
        if word.endswith(suffix) and len(word) > len(suffix) + 1:
            candidates.add(word[:-len(suffix)])
    
    # Strip both prefix and suffix
    for ps in prefix_stripped:
        for suffix in HEBREW_SUFFIXES:
            if ps.endswith(suffix) and len(ps) > len(suffix) + 1:
                candidates.add(ps[:-len(suffix)])
    
    return list(candidates)

def normalize_he(text):
    """
    Hebrew-specific normalization:
    - NFC unicode normalization
    - Strip niqqud (vowel diacritics U+05B0-U+05C7)
    - Strip apostrophes and edge punctuation
    - Strip Hebrew punctuation marks (geresh, gershayim)
    """
    text = unicodedata.normalize('NFC', str(text))
    # Strip Hebrew niqqud (combining diacritics)
    text = ''.join(c for c in text if not ('\u05B0' <= c <= '\u05C7'))
    # Strip apostrophes and curly quotes
    text = text.replace('\u2019', '').replace("'", '').replace('`', '')
    # Strip Hebrew punctuation marks (geresh ׳, gershayim ״)
    text = text.replace('\u05F3', '').replace('\u05F4', '')
    # Strip edge punctuation including Hebrew maqaf
    text = text.strip(' .,!?"()-:;[]{}\u05BE')
    return text.lower()


def strip_hebrew_prefix(word):
    """
    Try stripping common Hebrew prefixes to find the base form.
    Returns a list of candidate base forms (original + stripped versions).
    Longest prefixes are tried first to avoid over-stripping.
    Requires the remaining base to be at least 2 characters.
    """
    candidates = [word]
    for prefix in HEBREW_PREFIXES:
        if word.startswith(prefix) and len(word) > len(prefix) + 1:
            candidates.append(word[len(prefix):])
    return candidates


def count_occurrences_he(sentence_text, components):
    words = [normalize_he(w) for w in sentence_text.split()]
    words = [w for w in words if w]
    n = len(components)
    if n == 0 or len(words) < n:
        return 0
    count = 0
    i = 0
    while i <= len(words) - n:
        match = True
        for j in range(n):
            word = words[i + j]
            component = components[j]
            word_stems = get_hebrew_stem(word)
            comp_stems = get_hebrew_stem(component)
            found = any(ws == cs for ws in word_stems for cs in comp_stems)
            if not found:
                match = False
                break
        if match:
            count += 1
            i += n
        else:
            i += 1
    return count


def try_match_at_he_with_prefix(sentence_tokens, start, components,
                                  consumed_positions):
    phrase_len = len(components)
    if start + phrase_len > len(sentence_tokens):
        return False
    if set(range(start, start + phrase_len)) & consumed_positions:
        return False
    for i in range(phrase_len):
        token_text = sentence_tokens[start + i]['text']
        component = components[i]
        token_stems = get_hebrew_stem(token_text)
        comp_stems = get_hebrew_stem(component)
        if any(ts == cs for ts in token_stems for cs in comp_stems):
            continue
        return False
    return True


def get_sentence_tokens_he(sentence, tokenizer, model):
    """
    Run a Hebrew sentence through XLM-RoBERTa.
    Sentence passed EXACTLY as-is to RoBERTa (no normalization).
    normalize_he() applied only to text labels for matching.
    """
    encoded = tokenizer(sentence, return_tensors='pt',
                        truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**encoded)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    word_ids = encoded.word_ids()
    input_ids = encoded['input_ids'][0]

    word_vectors = {}
    word_token_ids = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_vectors:
            word_vectors[word_id] = []
            word_token_ids[word_id] = []
        word_vectors[word_id].append(token_embeddings[idx].numpy())
        word_token_ids[word_id].append(input_ids[idx].item())

    tokens = []
    for word_id in sorted(word_vectors.keys()):
        avg_vector = np.mean(word_vectors[word_id], axis=0)
        raw_text = tokenizer.decode(word_token_ids[word_id])
        norm_text = normalize_he(raw_text)
        if norm_text:
            tokens.append({'text': norm_text, 'vector': avg_vector})
    return tokens


def get_fallback_vector_he(he_word_text, tokenizer, model):
    """
    For words not found in any sentence, run the Hebrew word alone through RoBERTa.
    """
    encoded = tokenizer(he_word_text, return_tensors='pt',
                        truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**encoded)
    vecs = outputs.last_hidden_state.squeeze(0).numpy()
    word_ids = encoded.word_ids()
    valid_vecs = [vecs[i] for i, wid in enumerate(word_ids) if wid is not None]
    if valid_vecs:
        return np.mean(valid_vecs, axis=0)
    return np.zeros(768)


# Pre-compute normalized Hebrew components for all target words
he_target_components_list = []
for wi in range(len(word_level_df)):
    raw = str(word_level_df.iloc[wi]['he']).strip()
    components = [normalize_he(c) for c in raw.split(' ') if normalize_he(c)]
    he_target_components_list.append(components)

print('Helpers ready.')
print(f'\nSample Hebrew target components:')
for wi in range(10):
    print(f"  [{wi:3d}] '{word_level_df.iloc[wi]['he']}'  -> {he_target_components_list[wi]}")

Helpers ready.

Sample Hebrew target components:
  [  0] 'מערכה'  -> ['מערכה']
  [  1] 'קוף'  -> ['קוף']
  [  2] 'אמצע'  -> ['אמצע']
  [  3] 'מקומות'  -> ['מקומות']
  [  4] 'בעלי חיים'  -> ['בעלי', 'חיים']
  [  5] 'ללכת'  -> ['ללכת']
  [  6] 'מקומות'  -> ['מקומות']
  [  7] 'מעוצב'  -> ['מעוצב']
  [  8] 'בני אדם'  -> ['בני', 'אדם']
  [  9] 'בני אדם'  -> ['בני', 'אדם']


## 3. Assign Words to Sentences

In [3]:
print('Assigning Hebrew words to sentences...')

he_word_to_sentence = {}
he_unassignable = []
he_assignment_counts = defaultdict(lambda: defaultdict(int))

he_sent_idx = 0
n_he_sentences = len(he_sentences)

for wi in range(len(word_level_df)):
    components = he_target_components_list[wi]
    target_key = ' '.join(components)

    if not components:  # empty Hebrew entry
        he_unassignable.append(wi)
        he_word_to_sentence[wi] = None
        continue

    found = False
    search_end = min(he_sent_idx + 15, n_he_sentences)
    for candidate_idx in range(he_sent_idx, search_end):
        capacity = count_occurrences_he(he_sentences[candidate_idx], components)
        already_assigned = he_assignment_counts[candidate_idx][target_key]
        if capacity > already_assigned:
            he_word_to_sentence[wi] = candidate_idx
            he_assignment_counts[candidate_idx][target_key] += 1
            # Only advance pointer when sentence capacity is fully used
            if he_assignment_counts[candidate_idx][target_key] >= capacity:
                he_sent_idx = candidate_idx
            found = True
            break

    if not found:
        he_unassignable.append(wi)
        he_word_to_sentence[wi] = None

he_assigned = sum(1 for v in he_word_to_sentence.values() if v is not None)
print(f'Assigned     : {he_assigned} / {len(word_level_df)}')
print(f'Unassignable : {len(he_unassignable)}')

print('\nFirst 10 assignments:')
for wi in range(10):
    si = he_word_to_sentence.get(wi)
    he_word = word_level_df.iloc[wi]['he']
    sent_preview = he_sentences[si][:60] if si is not None else 'UNASSIGNED'
    print(f"  [{wi:3d}] '{he_word}'  -> sent {si}: '{sent_preview}'")

Assigning Hebrew words to sentences...
Assigned     : 97 / 1735
Unassignable : 1638

First 10 assignments:
  [  0] 'מערכה'  -> sent 0: 'מערכה ראשונה, קוף באמצע.'
  [  1] 'קוף'  -> sent 0: 'מערכה ראשונה, קוף באמצע.'
  [  2] 'אמצע'  -> sent 0: 'מערכה ראשונה, קוף באמצע.'
  [  3] 'מקומות'  -> sent 1: 'אז יש כמה מקומות שבעלי חיים כמעט ולא הולכים אליהם, מקומות שע'
  [  4] 'בעלי חיים'  -> sent 1: 'אז יש כמה מקומות שבעלי חיים כמעט ולא הולכים אליהם, מקומות שע'
  [  5] 'ללכת'  -> sent 6: 'הלכתי יום שלם רק בעקבותיהם לתוך עובי היער, דרך הסבך והגפנים '
  [  6] 'מקומות'  -> sent None: 'UNASSIGNED'
  [  7] 'מעוצב'  -> sent None: 'UNASSIGNED'
  [  8] 'בני אדם'  -> sent None: 'UNASSIGNED'
  [  9] 'בני אדם'  -> sent None: 'UNASSIGNED'


## 3b. Rescue Unassigned Words (Brute Force Full Scan)

In [4]:
print(f'Rescuing {len(he_unassignable)} unassigned words via full scan...\n')

# Build time map from successfully assigned words
he_sentence_time_map = {}
for wi, si in he_word_to_sentence.items():
    if si is None:
        continue
    t_start = word_level_df.iloc[wi]['start']
    t_end = word_level_df.iloc[wi]['end']
    if si not in he_sentence_time_map:
        he_sentence_time_map[si] = [t_start, t_end]
    else:
        he_sentence_time_map[si][0] = min(he_sentence_time_map[si][0], t_start)
        he_sentence_time_map[si][1] = max(he_sentence_time_map[si][1], t_end)

he_rescued = 0
he_still_unassignable = []

for wi in he_unassignable:
    components = he_target_components_list[wi]
    target_key = ' '.join(components)
    t = word_level_df.iloc[wi]['start']

    if not components:
        he_still_unassignable.append(wi)
        continue

    candidates = []
    for si in range(n_he_sentences):
        capacity = count_occurrences_he(he_sentences[si], components)
        remaining = capacity - he_assignment_counts[si][target_key]
        if remaining > 0:
            if si in he_sentence_time_map:
                proximity = abs(he_sentence_time_map[si][0] - t)
            else:
                proximity = 9999
            candidates.append((proximity, si))

    if candidates:
        candidates.sort()
        best_si = candidates[0][1]
        he_word_to_sentence[wi] = best_si
        he_assignment_counts[best_si][target_key] += 1
        he_rescued += 1
    else:
        he_still_unassignable.append(wi)

print(f'Rescued via full scan : {he_rescued}')
print(f'True unassignable     : {len(he_still_unassignable)}')
he_unassignable = he_still_unassignable

if he_still_unassignable:
    print(f'\nTrue unassignable (will get fallback vectors):')
    for wi in he_still_unassignable:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  en='{row['en']}'  he='{row['he']}'")

Rescuing 1638 unassigned words via full scan...

Rescued via full scan : 1152
True unassignable     : 486

True unassignable (will get fallback vectors):
  [  23] t=28.4s  en='traveled'  he='טייל'
  [  29] t=32.0s  en='gone'  he='נעלם'
  [  33] t=34.1s  en='thick'  he='עבה'
  [  47] t=52.6s  en='met'  he='נפגש'
  [  50] t=54.0s  en='mean'  he='ממוצע'
  [  54] t=56.4s  en='little_bit'  he='קצת קצת'
  [  58] t=58.6s  en='showing'  he='הצגה'
  [  60] t=59.9s  en='show'  he='להציג'
  [  62] t=60.7s  en='stare'  he='לנעוץ מבט'
  [  65] t=64.8s  en='able'  he='מסוגל'
  [  71] t=69.5s  en='shot'  he='בעיטה'
  [  78] t=76.5s  en='taking'  he='לקיחה'
  [  79] t=77.0s  en='shot'  he='בעיטה'
  [  84] t=81.8s  en='changed'  he='השתנה'
  [  86] t=83.6s  en='put'  he='לשים'
  [  94] t=88.8s  en='hoping'  he='מקווה'
  [  98] t=91.3s  en='pushing'  he='דוחף'
  [  99] t=94.1s  en='curious'  he='סקרן'
  [ 100] t=96.0s  en='pushing'  he='דוחף'
  [ 101] t=99.0s  en='attached'  he='מצורף'
  [ 109] t=105.2s

##  3c. SECOND RESCUE: ignore capacity, find closest sentence by timestamp 


In [5]:
print(f"Second rescue pass for {len(he_unassignable)} remaining words...\n")

he_rescued_2 = 0
he_still_unassignable_2 = []

for wi in he_unassignable:
    components = he_target_components_list[wi]
    target_key = ' '.join(components)
    t = word_level_df.iloc[wi]['start']

    if not components:
        he_still_unassignable_2.append(wi)
        continue

    # Search ALL sentences ignoring capacity — just find closest by timestamp
    # that actually contains the word in any form
    candidates = []
    for si in range(n_he_sentences):
        sent_words = [normalize_he(w) for w in he_sentences[si].split() 
                      if normalize_he(w)]
        
        # Check if all components appear in sentence (with prefix stripping)
        all_found = True
        for comp in components:
            comp_candidates = strip_hebrew_prefix(comp)
            word_found = False
            for sw in sent_words:
                sw_candidates = strip_hebrew_prefix(sw)
                if any(c == s for c in comp_candidates for s in sw_candidates):
                    word_found = True
                    break
            if not word_found:
                all_found = False
                break
        
        if all_found:
            if si in he_sentence_time_map:
                proximity = abs(he_sentence_time_map[si][0] - t)
            else:
                proximity = 9999
            candidates.append((proximity, si))

    if candidates:
        candidates.sort()
        best_si = candidates[0][1]
        he_word_to_sentence[wi] = best_si
        # Don't update assignment_counts — we're ignoring capacity here
        he_rescued_2 += 1
    else:
        he_still_unassignable_2.append(wi)

print(f"Rescued in second pass : {he_rescued_2}")
print(f"Still unassignable     : {len(he_still_unassignable_2)}")
he_unassignable = he_still_unassignable_2

if he_still_unassignable_2:
    print(f"\nRemaining true unassignables (different vocabulary):")
    for wi in he_still_unassignable_2[:30]:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  en='{row['en']}'  he='{row['he']}'")

Second rescue pass for 486 remaining words...

Rescued in second pass : 113
Still unassignable     : 373

Remaining true unassignables (different vocabulary):
  [  23] t=28.4s  en='traveled'  he='טייל'
  [  29] t=32.0s  en='gone'  he='נעלם'
  [  33] t=34.1s  en='thick'  he='עבה'
  [  47] t=52.6s  en='met'  he='נפגש'
  [  50] t=54.0s  en='mean'  he='ממוצע'
  [  58] t=58.6s  en='showing'  he='הצגה'
  [  60] t=59.9s  en='show'  he='להציג'
  [  62] t=60.7s  en='stare'  he='לנעוץ מבט'
  [  65] t=64.8s  en='able'  he='מסוגל'
  [  71] t=69.5s  en='shot'  he='בעיטה'
  [  78] t=76.5s  en='taking'  he='לקיחה'
  [  79] t=77.0s  en='shot'  he='בעיטה'
  [  84] t=81.8s  en='changed'  he='השתנה'
  [  86] t=83.6s  en='put'  he='לשים'
  [  94] t=88.8s  en='hoping'  he='מקווה'
  [  98] t=91.3s  en='pushing'  he='דוחף'
  [ 100] t=96.0s  en='pushing'  he='דוחף'
  [ 101] t=99.0s  en='attached'  he='מצורף'
  [ 109] t=105.2s  en='putting'  he='לשים'
  [ 111] t=106.3s  en='fighting'  he='לחימה'
  [ 120] t=117

## 4. Load Model

In [6]:
print('Loading XLM-RoBERTa...')
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()
print('Model ready.')

Loading XLM-RoBERTa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]Error processing line 1 of /Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site.py", line 195, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9082.63it/s]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

Model ready.


## 5. Extraction Loop

In [7]:
print('Extracting Hebrew embeddings...\n')

he_final_embeddings = []
he_matched_indices = []
he_fallback_indices = []
he_sentence_token_cache = {}

def get_he_tokens_cached(si):
    if si not in he_sentence_token_cache:
        he_sentence_token_cache[si] = get_sentence_tokens_he(
            he_sentences[si], tokenizer, model
        )
    return he_sentence_token_cache[si]

# Group words by assigned sentence
he_words_per_sentence = defaultdict(list)
for wi, si in he_word_to_sentence.items():
    if si is not None:
        he_words_per_sentence[si].append(wi)

# Process sentence by sentence
for si in sorted(he_words_per_sentence.keys()):
    target_word_indices = sorted(he_words_per_sentence[si])
    sentence_tokens = get_he_tokens_cached(si)
    n_tokens = len(sentence_tokens)

    consumed_positions = set()
    sentence_pointer = 0

    for word_idx in target_word_indices:
        components = he_target_components_list[word_idx]
        phrase_len = len(components)

        def try_match_at(start):
            return try_match_at_he_with_prefix(
                sentence_tokens, start, components, consumed_positions
            )

        # Pass 1: forward from current pointer
        matched_at = None
        for start in range(sentence_pointer, n_tokens):
            if try_match_at(start):
                matched_at = start
                break

        # Pass 2: backward (handles repeated words)
        if matched_at is None:
            for start in range(0, sentence_pointer):
                if try_match_at(start):
                    matched_at = start
                    break

        if matched_at is not None:
            phrase_vectors = [
                sentence_tokens[matched_at + i]['vector']
                for i in range(phrase_len)
            ]
            he_final_embeddings.append(np.mean(phrase_vectors, axis=0))
            he_matched_indices.append(word_idx)
            for i in range(phrase_len):
                consumed_positions.add(matched_at + i)
            if matched_at >= sentence_pointer:
                sentence_pointer = matched_at + phrase_len
        else:
            # Assigned to sentence but token match failed -> fallback
            raw_he = str(word_level_df.iloc[word_idx]['he']).strip()
            vec = get_fallback_vector_he(raw_he, tokenizer, model)
            he_final_embeddings.append(vec)
            he_matched_indices.append(word_idx)
            he_fallback_indices.append(word_idx)

# Fallback for all truly unassignable words
for wi in he_unassignable:
    raw_he = str(word_level_df.iloc[wi]['he']).strip()
    vec = get_fallback_vector_he(raw_he, tokenizer, model)
    he_final_embeddings.append(vec)
    he_matched_indices.append(wi)
    he_fallback_indices.append(wi)

# Sort back into word_level_df order
he_combined = sorted(
    zip(he_matched_indices, he_final_embeddings),
    key=lambda x: x[0]
)
he_matched_sorted = [x[0] for x in he_combined]
he_embeddings_sorted = [x[1] for x in he_combined]

print('Extraction complete.')

Extracting Hebrew embeddings...

Extraction complete.


## 6. Report

In [8]:
print('=' * 55)
print('HEBREW ALIGNMENT REPORT')
print('=' * 55)
print(f'Total target words    : {len(word_level_df)}')
print(f'Full context matches  : {len(he_matched_indices) - len(he_fallback_indices)}')
print(f'Fallback vectors      : {len(he_fallback_indices)}')
print(f'Total vectors         : {len(he_embeddings_sorted)}')
match_rate = (len(he_matched_indices) - len(he_fallback_indices)) / len(word_level_df) * 100
print(f'Full context rate     : {match_rate:.1f}%')

# Verify perfect alignment
assert len(he_embeddings_sorted) == len(word_level_df), \
    f'ALIGNMENT ERROR: {len(he_embeddings_sorted)} vectors != {len(word_level_df)} words'
assert he_matched_sorted == list(range(len(word_level_df))), \
    'ORDER ERROR: indices not 0..1734'

print('\nVerified: all 1735 Hebrew words have exactly one vector.')

if he_fallback_indices:
    print(f'\nFirst 20 fallback words:')
    for idx in sorted(he_fallback_indices)[:20]:
        row = word_level_df.iloc[idx]
        print(f"  [{idx:4d}] t={row['start']:.1f}s  en='{row['en']}'  he='{row['he']}'")

HEBREW ALIGNMENT REPORT
Total target words    : 1735
Full context matches  : 1045
Fallback vectors      : 690
Total vectors         : 1735
Full context rate     : 60.2%

Verified: all 1735 Hebrew words have exactly one vector.

First 20 fallback words:
  [  23] t=28.4s  en='traveled'  he='טייל'
  [  29] t=32.0s  en='gone'  he='נעלם'
  [  33] t=34.1s  en='thick'  he='עבה'
  [  41] t=42.6s  en='man'  he='אדם'
  [  47] t=52.6s  en='met'  he='נפגש'
  [  50] t=54.0s  en='mean'  he='ממוצע'
  [  54] t=56.4s  en='little_bit'  he='קצת קצת'
  [  58] t=58.6s  en='showing'  he='הצגה'
  [  60] t=59.9s  en='show'  he='להציג'
  [  62] t=60.7s  en='stare'  he='לנעוץ מבט'
  [  65] t=64.8s  en='able'  he='מסוגל'
  [  71] t=69.5s  en='shot'  he='בעיטה'
  [  78] t=76.5s  en='taking'  he='לקיחה'
  [  79] t=77.0s  en='shot'  he='בעיטה'
  [  84] t=81.8s  en='changed'  he='השתנה'
  [  86] t=83.6s  en='put'  he='לשים'
  [  94] t=88.8s  en='hoping'  he='מקווה'
  [  98] t=91.3s  en='pushing'  he='דוחף'
  [  99] 

## 7. Save

In [9]:
# Save embeddings
he_embeddings_df = pd.DataFrame(he_embeddings_sorted)
out_emb = '../data/processed/he_contextual_aligned_embeddings.csv'
he_embeddings_df.to_csv(out_emb, index=False)

# Save quality flags
he_quality_df = pd.DataFrame({
    'word_idx': list(range(len(word_level_df))),
    'en': word_level_df['en'].tolist(),
    'he': word_level_df['he'].tolist(),
    'is_fallback': [i in set(he_fallback_indices) for i in range(len(word_level_df))]
})
he_quality_df.to_csv('../data/processed/he_contextual_quality_flags.csv', index=False)

print(f'Saved embeddings  -> {out_emb}')
print(f'Shape             : {he_embeddings_df.shape}  (expected: 1735 x 768)')
print(f'Any NaN           : {he_embeddings_df.isnull().any().any()}')
print(f'\nFull context      : {(~he_quality_df["is_fallback"]).sum()}')
print(f'Fallback          : {he_quality_df["is_fallback"].sum()}')
print(f'\nSample row 0 (first 5 dims): {he_embeddings_df.iloc[0, :5].tolist()}')

Saved embeddings  -> ../data/processed/he_contextual_aligned_embeddings.csv
Shape             : (1735, 768)  (expected: 1735 x 768)
Any NaN           : False

Full context      : 1045
Fallback          : 690

Sample row 0 (first 5 dims): [0.06651099771261215, 0.00244021974503994, -0.022181615233421326, 0.007284244522452354, 0.14750196039676666]


In [10]:
# Check a sample of the 712 unassigned words against ALL sentences
print("Checking if unassigned words exist ANYWHERE in the Hebrew sentences...\n")

# Get the words that have no sentence assigned
no_sent_words = [wi for wi in sorted(he_fallback_indices) 
                 if he_word_to_sentence.get(wi) is None]

print(f"Total with no sentence: {len(no_sent_words)}")
print(f"\nChecking first 20 against all sentences:")

for word_idx in no_sent_words[:20]:
    row = word_level_df.iloc[word_idx]
    components = he_target_components_list[word_idx]
    
    # Search all sentences for any partial match
    found_in = []
    for si, sent in enumerate(he_sentences):
        sent_words = [normalize_he(w) for w in sent.split() if normalize_he(w)]
        for comp in components:
            comp_candidates = strip_hebrew_prefix(comp)
            for sw in sent_words:
                sw_candidates = strip_hebrew_prefix(sw)
                if any(c == s for c in comp_candidates for s in sw_candidates):
                    found_in.append(si)
                    break
    
    found_in = list(set(found_in))
    print(f"  [{word_idx:4d}] en='{row['en']}'  he='{row['he']}'")
    print(f"         found in sentences: {found_in[:5] if found_in else 'NOWHERE'}")

Checking if unassigned words exist ANYWHERE in the Hebrew sentences...

Total with no sentence: 373

Checking first 20 against all sentences:
  [  23] en='traveled'  he='טייל'
         found in sentences: NOWHERE
  [  29] en='gone'  he='נעלם'
         found in sentences: NOWHERE
  [  33] en='thick'  he='עבה'
         found in sentences: NOWHERE
  [  47] en='met'  he='נפגש'
         found in sentences: NOWHERE
  [  50] en='mean'  he='ממוצע'
         found in sentences: NOWHERE
  [  58] en='showing'  he='הצגה'
         found in sentences: NOWHERE
  [  60] en='show'  he='להציג'
         found in sentences: NOWHERE
  [  62] en='stare'  he='לנעוץ מבט'
         found in sentences: [144]
  [  65] en='able'  he='מסוגל'
         found in sentences: NOWHERE
  [  71] en='shot'  he='בעיטה'
         found in sentences: NOWHERE
  [  78] en='taking'  he='לקיחה'
         found in sentences: NOWHERE
  [  79] en='shot'  he='בעיטה'
         found in sentences: NOWHERE
  [  84] en='changed'  he='השתנה'
  